https://research-information.bris.ac.uk/ws/portalfiles/portal/395960811/thesis.pdf 
Eddie Jones thesis

cyclic proofs

Proof by saturation


https://dl.acm.org/doi/10.1007/978-3-031-99984-0_8  smt verificatio nof vampire proofs?
```
--print_proofs_to_file (-pptf)
--
	for proof-checking by external solvers
	- tptp gives TPTP output
	- property is a developmental option. It allows developers to output statistics 
	about the proof using a ProofPrinter object (see Kernel/InferenceStore::ProofPropertyPrinter
	- smtcheck produces a ground SMT script for proof checking
	
	default: on
	values: off,on,proofcheck,tptp,property,smt2_proofcheck,smtcheck
```

agda proof reconstruction
https://arxiv.org/abs/2602.18844 when agda met vampire

https://arxiv.org/abs/2602.16324 Case Study: Saturations as Explicit Models in Equational Theories

Rawson
https://people.ciirc.cvut.cz/~janotmik/ Mikoláš Janota

https://arxiv.org/abs/1607.04829  Logic Programming with Graph Automorphism: Integrating naut with Prolog (Tool Description)
https://github.com/michael-123123/plnauty
https://github.com/michael-123123/plcliquer

https://arxiv.org/abs/2506.02903 symmettry breaking papers
https://dl.acm.org/doi/10.1609/aaai.v38i8.28643 SAT based techniques for minimal graphs

Michael Codish

I should ask questions on vampire issues?


satrurated clause sets exhibit models as explicit indcutively defined subsets of the herbrand universe.
The intersection of all models using the clauses oreinted into rules according to the order. But we didn't know there were any models?

Rewrites do the same thing


In [ ]:
from kdrag.all import *

G = smt.DeclareSort("G")
mul = smt.Function("mul", G, G, G)
inv = smt.Function("inv", G, G)
e = smt.Const("e", G)
kd.notation.mul.register(G, mul)
kd.notation.invert.register(G, inv)
x,y,z = smt.Consts("x y z", G)

mul_assoc = kd.axiom(smt.ForAll([x,y,z], mul(x, mul(y,z)) == mul(mul(x,y), z)))
mul_inv = kd.axiom(smt.ForAll([x], mul(inv(x), x) == e))
mul_e_l = kd.axiom(smt.ForAll([x], mul(e, x) == x))
mul_e_r = kd.axiom(smt.ForAll([x], mul(x, e) == x))

pf = kd.tactics.vprove(smt.ForAll([x], mul(x, inv(x)) == e), by=[mul_assoc, mul_inv, mul_e_l, mul_e_r])
#kd.prove(smt.ForAll([x], mul(x, inv(x)) == e), by=[mul_assoc, mul_inv, mul_e_l, mul_e_r]) # a bit unstable




['prove',
 |= ForAll([x, y, z], mul(x, mul(y, z)) == mul(mul(x, y), z)),
 |= ForAll(x, mul(inv(x), x) == e),
 |= ForAll(x, mul(e, x) == x),
 |= ForAll(x, mul(x, e) == x)]

In [ ]:
%%file /tmp/group_sat.p

cnf(ax1, axiom, mul(X,mul(Y,Z)) = mul(mul(X, Y), Z)).
cnf(ax2, axiom, mul(inv(X), X) = e).
cnf(ax4, axiom, mul(X,inv(X)) = e).
cnf(ax3, axiom, mul(e, X) = X).
cnf(ax4, axiom, mul(X, e) = X).

Writing /tmp/group_sat.p


In [57]:
! vampire /tmp/group_sat.p --mode casc  --intent sat 

% Running in auto input_syntax mode. Trying TPTP
% Will run a generic schedule for satisfiability detection.
% WARNING: time unlimited strategy and instruction limiting not in place - attempting to translate instructions to time
% dis+10_1_sil=32000:sp=arity:random_seed=3527647588:i=103:fgj=on_1 on group_sat for (1ds/103Mi)
% Time limit reached! 
% ------------------------------
% Version: Vampire 5.0.1 (Release build, commit 1b13eaf on 2026-01-18 12:14:50 +0000)
% Linked with Z3 4.14.0.0 3c47fd96cf5645d0c42b2c819d9e9a84380aa721 NOTFOUND
% CaDiCaL version: 2.1.3
% Termination reason: Time limit
% Termination phase: Saturation
% Time elapsed: 0.100 s
% Peak memory usage: 21 MB
% ------------------------------
% ------------------------------
% WARNING: time unlimited strategy and instruction limiting not in place - attempting to translate instructions to time
% ott+31_1_sil=16000:lcm=predicate:bce=on:newcnf=on:random_seed=3495493205:i=116_1 on group_sat for (1ds/116Mi)
% Time limit reac

In [58]:
%%file /tmp/group.p

cnf(ax1, axiom, mul(X,mul(Y,Z)) = mul(mul(X, Y), Z)).
cnf(ax2, axiom, mul(inv(X), X) = e).
%cnf(ax4, axiom, mul(X,inv(X)) = e).
cnf(ax3, axiom, mul(e, X) = X).
cnf(ax4, axiom, mul(X,e) = X).
cnf(ax5, axiom, mul(X,inv(X)) != e).


Overwriting /tmp/group.p


In [49]:
! vampire /tmp/group.p --mode vampire --proof tptp

% Running in auto input_syntax mode. Trying TPTP
% Refutation found. Thanks to Tanya!
% SZS status Unsatisfiable for group
% SZS output start Proof for group
fof(f2,axiom,(
  ( ! [X0] : (mul(inv(X0),X0) = e) )),
  file('/tmp/group.p',unknown)).
fof(f3,axiom,(
  ( ! [X0] : (mul(e,X0) = X0) )),
  file('/tmp/group.p',unknown)).
fof(f4,axiom,(
  ( ! [X0] : (mul(X0,e) = X0) )),
  file('/tmp/group.p',unknown)).
fof(f5,axiom,(
  ( ! [X0] : (mul(X0,inv(X0)) != e) )),
  file('/tmp/group.p',unknown)).
fof(f6,plain,(
  ( ! [X0] : (e != mul(X0,inv(X0))) )),
  inference(reorient_equations,[],[f5])).
fof(f8,plain,(
  e = inv(e)),
  inference(superposition,[],[f4,f2])).
fof(f9,plain,(
  e != inv(e)),
  inference(superposition,[],[f6,f3])).
fof(f10,plain,(
  $false),
  inference(forward_subsumption_resolution,[],[f9,f8])).
% SZS output end Proof for group
% ------------------------------
% Version: Vampire 5.0.1 (Release build, commit 1b13eaf on 2026-01-18 12:14:50 +0000)
% Linked with Z3 4.14.0.0 3c4

In [47]:
! vampire /tmp/group.p --mode vampire  --intent sat --proof smtcheck --proof_extra full

% Running in auto input_syntax mode. Trying TPTP
% Refutation found. Thanks to Tanya!
% SZS status Unsatisfiable for group
% SZS output start Proof for group
(set-logic ALL)
(declare-sort iota 0)
(declare-const inhabit_iota iota)
(declare-const inhabit_Int Int)
(declare-const inhabit_Real Real)
(declare-const inhabit_Bool Bool)
(declare-fun is_rat (Real) Bool)
(declare-fun |_mul| (iota iota) iota)
(declare-fun |_inv| (iota) iota)
(declare-fun |_e| () iota)

; step 6
(push)
(declare-const v0 iota)
(echo "sorry: reorient equations")
(pop)

; step 8
(push)
(assert (or (= (|_inv| |_e|) (|_mul| (|_inv| |_e|) |_e|))))
(assert (or (= |_e| (|_mul| (|_inv| |_e|) |_e|))))
(assert (not (= |_e| (|_inv| |_e|))))
(set-info :status unsat)
(check-sat)
(pop)

; step 9
(push)
(assert (or (not (= |_e| (|_mul| |_e| (|_inv| |_e|))))))
(assert (or (= (|_inv| |_e|) (|_mul| |_e| (|_inv| |_e|)))))
(assert (= |_e| (|_inv| |_e|)))
(set-info :status unsat)
(check-sat)
(pop)

; step 10
(push)
(assert (or (not (= |

In [35]:
! eprover-ho /tmp/group.p --proof-object=3

# Initializing proof state
# Scanning for AC axioms
# mul is associative
#
#cnf(i_0_9, plain, (mul(X1,e)=X1)).
#
#cnf(i_0_8, plain, (mul(e,X1)=X1)).
#
#cnf(i_0_7, plain, (mul(inv(X1),X1)=e)).
#
#cnf(i_0_10, plain, (mul(X1,inv(X1))!=e)).
#
#cnf(i_0_11, plain, (inv(e)=e)).

# Proof found!
# SZS status Unsatisfiable
# SZS output start CNFRefutation
cnf(ax5, axiom, (mul(X1,inv(X1))!=e), file('/tmp/group.p', ax5)).
cnf(ax4, axiom, (mul(X1,e)=X1), file('/tmp/group.p', ax4)).
cnf(ax2, axiom, (mul(inv(X1),X1)=e), file('/tmp/group.p', ax2)).
cnf(c_0_3, plain, (mul(X1,inv(X1))!=e), inference(fof_simplification,[status(thm)],[ax5])).
cnf(c_0_4, axiom, (mul(X1,e)=X1), ax4).
cnf(c_0_5, axiom, (mul(inv(X1),X1)=e), ax2).
cnf(c_0_6, plain, (mul(X1,inv(X1))!=e), c_0_3).
cnf(c_0_7, plain, (inv(e)=e), inference(pm,[status(thm)],[c_0_4, c_0_5])).
cnf(c_0_8, plain, ($false), inference(cn,[status(thm)],[inference(rw,[status(thm)],[inference(pm,[status(thm)],[c_0_6, c_0_7]), c_0_4])]), ['proof']).
# SZS outp

In [40]:
! eprover-ho --print-oriented-eqlits-as-rules /tmp/group_sat.p  --print-saturated --presat-simplify

% (lift_lambdas = 1, lambda_to_forall = 1,unroll_only_formulas = 1, sine = (null))
% Initializing proof state
% Scanning for AC axioms
% mul is associative
%
%cnf(i_0_10, plain, (mul(X1,e)->X1)).
%
%cnf(i_0_9, plain, (mul(e,X1)->X1)).
%
%cnf(i_0_8, plain, (mul(X1,inv(X1))->e)).
%
%cnf(i_0_7, plain, (mul(inv(X1),X1)->e)).
%
%cnf(i_0_6, plain, (mul(mul(X1,X2),X3)->mul(X1,mul(X2,X3)))).
% Presaturation interreduction done
%
%cnf(i_0_10, plain, (mul(X1,e)->X1)).
%
%cnf(i_0_9, plain, (mul(e,X1)->X1)).
%
%cnf(i_0_8, plain, (mul(X1,inv(X1))->e)).
%
%cnf(i_0_11, plain, (inv(e)->e)).
%
%cnf(i_0_7, plain, (mul(inv(X1),X1)->e)).
%
%cnf(i_0_6, plain, (mul(mul(X1,X2),X3)->mul(X1,mul(X2,X3)))).
%
%cnf(i_0_19, plain, (mul(inv(X1),mul(X1,X2))->X2)).
%
%cnf(i_0_25, plain, (inv(inv(X1))->X1)).
%
%cnf(i_0_21, plain, (mul(X1,mul(inv(X1),X2))->X2)).
%%%
%cnf(i_0_16, plain, (mul(X1,mul(X2,inv(mul(X1,X2))))->e)).
%
%cnf(i_0_49, plain, (mul(X2,inv(mul(X1,X2)))->inv(X1))).
%
%cnf(i_0_65, plain, (mul(inv(mul(X1

Not sure presat simpkuft did anything

In [16]:
import janus_swi as janus
import subprocess

res = subprocess.run(["eprover-ho", "--print-oriented-eqlits-as-rules", "/tmp/group_sat.p", "--print-saturated", "--presat-simplify", "-o", "/tmp/eprover.p"])
#print(res.stdout)
"""
janus.consult("", 
       res.stdout,
       module="eprover"
              )
"""
janus.consult("/tmp/group.p")
#janus.consult("/tmp/eprover.p")
list(janus.query("cnf(_X,_Y,_Z), X = prolog(_X), Y = prolog(_Y), Z = prolog(_Z)."))


% (lift_lambdas = 1, lambda_to_forall = 1,unroll_only_formulas = 1, sine = (null))


ERROR: [Thread main] /tmp/group.p:7:30: Syntax error: Operator expected


[{'truth': True,
  'X': ax1,
  'Y': axiom,
  'Z': =(mul(A,mul(B,C)),mul(mul(A,B),C))},
 {'truth': True, 'X': ax2, 'Y': axiom, 'Z': =(mul(inv(A),A),e)},
 {'truth': True, 'X': ax3, 'Y': axiom, 'Z': =(mul(e,A),A)},
 {'truth': True, 'X': ax4, 'Y': axiom, 'Z': =(mul(A,e),A)}]

In [25]:
import scryer
m = scryer.Machine()

m.load_module_filename("foo", "/tmp/group.p")
m.query_all("foo:cnf(X,Y,Z).")

RuntimeError: Prolog error term: error(existence_error(procedure, :(foo, /(cnf, 3))), /(cnf, 3))